Sentiment Analysis following a Tutorial

This practice project follows a tutorial that builds a sentiment analysis model to classify text into three sentiment categories: neutral, positive, and negative. The process includes data preprocessing, feature extraction via vectorization, model training with logistic regression, and evaluation using various classification metrics.

In [32]:
#downloading required modules
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer

In [54]:
data = pd.read_csv('sentiment_analysis.csv')
data.shape

(499, 7)

In [64]:
data.head(20)

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram
5,2016,11,22,night,"Its night 2 am, feeling neutral",neutral,Facebook
6,2017,12,28,morning,2 am feedings for the baby are fun when he is ...,positive,Facebook
7,2017,12,28,noon,Soooo high,neutral,Instagram
8,2019,10,28,night,Both of you,neutral,Twitter
9,2018,5,28,morning,Today first time I arrive in the boat. Its ama...,positive,Facebook


In [63]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           499 non-null    int64 
 1   Month          499 non-null    int64 
 2   Day            499 non-null    int64 
 3   Time of Tweet  499 non-null    object
 4   text           499 non-null    object
 5   sentiment      499 non-null    object
 6   Platform       499 non-null    object
dtypes: int64(3), object(4)
memory usage: 27.4+ KB


Dataset preprocessing: selecting feature & label, train-test split. A must every time you start a project

In [26]:
X = data["text"]
y = data["sentiment"]

In [35]:
data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.398798
positive    0.332665
negative    0.268537
Name: proportion, dtype: float64

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42) #I added stratification to get a more accurate data representation

In [41]:
y_train.value_counts(normalize=True)

sentiment
neutral     0.398496
positive    0.333333
negative    0.268170
Name: proportion, dtype: float64

Text vectorization: Used CountVectorizer to convert text data into a bag-of-words numerical format. Models like logistic regression can't understand text; they need it in a numerical format. Count Vectorizer just gives a sparse matrix of the counts of specific words (tokenizes and makes everything lowercase so no need to do that yourself)

In [42]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

Model Training: Used logistic regression (also used SVC in a Kaggle dataset). Because we're feeding it a sparse matrix, zero values aren't computed, making it very fast and efficient.

In [43]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)

LogisticRegression()

Model Evaluation:

In [44]:
y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy) # the number of correct preditions to the total predicitons.

Accuracy: 0.65


In [45]:
print(confusion_matrix(y_test, y_pred))

[[10 15  2]
 [ 2 33  5]
 [ 2  9 22]]


In [46]:
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

    negative       0.71      0.37      0.49        27
     neutral       0.58      0.82      0.68        40
    positive       0.76      0.67      0.71        33

    accuracy                           0.65       100
   macro avg       0.68      0.62      0.63       100
weighted avg       0.67      0.65      0.64       100



Model performs best on positive sentiment and struggles more with neutral/negative sentiment. Recall for neutral sentiment suggests that it captures most neutral cases but confuses them with other classes often. Very low recall for negative sentiment, maybe use oversampling?

Potential causes for mediocre performance: Low amounts of data processing, very small datset, maybe model itself isn't sufficient (Although the SVC I used on a diffeent dataset gave me similar results. Maybe I can use an SGDClassifier?)

Potential improvements: Use tfidvectorizer, use more preprocessing techniques (punctuation removal, lemmatization, stopword removal), explore more advanced models, get a much larger model since this one has only got 499 instances

In [10]:
#Decided to try using tf-id vectorization since
#I don't have much experience with other preprocessing techniques

In [11]:
X2 = data['text']
y2 = data['sentiment']
X2 = X2.str.lower()
X2 = X2.astype(str)

In [47]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, stratify=y, random_state=42)

In [55]:
vectorizer2 = TfidfVectorizer()
X2_train_vec = vectorizer2.fit_transform(X2_train)
X2_test_vec = vectorizer2.transform(X2_test)

In [56]:
model2 = LogisticRegression()
model2.fit(X2_train_vec, y2_train)

LogisticRegression()

In [57]:
y2_pred = model2.predict(X2_test_vec)

In [58]:
print("Accuracy Score:", accuracy_score(y2_test, y2_pred))
#higher accuracy score

Accuracy Score: 0.69


In [60]:
print("Classification Report:")
print(classification_report(y2_test, y2_pred)) #better results than the CountVectorizer! (shown below for comparison)

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.41      0.54        27
     neutral       0.61      0.88      0.72        40
    positive       0.79      0.70      0.74        33

    accuracy                           0.69       100
   macro avg       0.73      0.66      0.67       100
weighted avg       0.72      0.69      0.68       100



In [53]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.71      0.37      0.49        27
     neutral       0.58      0.82      0.68        40
    positive       0.76      0.67      0.71        33

    accuracy                           0.65       100
   macro avg       0.68      0.62      0.63       100
weighted avg       0.67      0.65      0.64       100

